# E1--E4 manuscript plotting

This notebook reads the frozen `results/` artifacts and regenerates every
manuscript metric, density, and scatter figure. It never reruns a sampler.

Outputs are written in four publication formats, one directory per format:
`figures/png/` (600 dpi), `figures/tiff/` (600 dpi, LZW), `figures/svg/`, and
`figures/pdf/`. BAOAB is labelled **ULD** in all figures.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if not (ROOT / "src" / "manuscript.py").is_file():
    raise RuntimeError("Run this notebook from the repository's notebooks/ directory")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))
sys.path.insert(0, str(ROOT))

from src.manuscript import EXPERIMENTS, METRICS, RESOURCE_AXES

print("Project root:", ROOT)
for key, spec in EXPERIMENTS.items():
    print(spec.number, key, "->", [spec.display_labels[m] for m in spec.methods])
print("Metrics:", METRICS)
print("Resource axes:", RESOURCE_AXES)

Project root: /home/zheyuanlai/levy-sampling-production
E1 double_well -> ['ULA', 'ULD', 'PT', 'FLA', 'Raw-CP', 'LSC-CP', 'LSC-CP-RA']
E2 mog40 -> ['ULA', 'ULD', 'PT', 'FLA', 'LSC-CP', 'LSC-CP-RA']
E3 mb3well_10d -> ['ULA', 'ULD', 'PT', 'FLA', 'LSC-CP', 'LSC-CP-RA (4)']
E4 coupled_phi4 -> ['ULA', 'ULD', 'PT', 'FLA', 'LSC-CP', 'LSC-CP-RA (8)']
Metrics: ('W2', 'MMD', 'TV', 'worst_basin_ESS')
Resource axes: ('t', 'nfe', 'wallclock')


In [2]:
from scripts.validate_release import validate_release

validate_release(ROOT, check_results=True, require_figures=False)
print("Frozen inputs are complete.")

Frozen inputs are complete.


In [3]:
metric_command = [
    sys.executable,
    str(ROOT / "scripts" / "replot_manuscript_figures.py"),
    "--results-dir", str(ROOT / "results"),
    "--figures-dir", str(ROOT / "figures"),
    "--no-clean",
]
print("Running:", " ".join(metric_command))
subprocess.run(metric_command, cwd=ROOT, env=os.environ.copy(), check=True)

Running: /home/zheyuanlai/miniconda3/envs/jcp-levy-release/bin/python /home/zheyuanlai/levy-sampling-production/scripts/replot_manuscript_figures.py --results-dir /home/zheyuanlai/levy-sampling-production/results --figures-dir /home/zheyuanlai/levy-sampling-production/figures --no-clean


Regenerated manuscript figures: 60 PNG, 60 TIFF, 60 SVG, 60 PDF in /home/zheyuanlai/levy-sampling-production/figures


CompletedProcess(args=['/home/zheyuanlai/miniconda3/envs/jcp-levy-release/bin/python', '/home/zheyuanlai/levy-sampling-production/scripts/replot_manuscript_figures.py', '--results-dir', '/home/zheyuanlai/levy-sampling-production/results', '--figures-dir', '/home/zheyuanlai/levy-sampling-production/figures', '--no-clean'], returncode=0)

In [4]:
sample_command = [
    sys.executable,
    str(ROOT / "scripts" / "replot_generated_samples.py"),
    "--results-root", str(ROOT / "results"),
    "--output-root", str(ROOT / "figures"),
    "--cache-root", str(ROOT / "cache" / "generated_samples"),
    "--manifest-path",
    str(ROOT / "cache" / "generated_samples" / "generated_sample_plots_manifest.json"),
    "--overwrite",
]
print("Running:", " ".join(sample_command))
subprocess.run(sample_command, cwd=ROOT, env=os.environ.copy(), check=True)

Running: /home/zheyuanlai/miniconda3/envs/jcp-levy-release/bin/python /home/zheyuanlai/levy-sampling-production/scripts/replot_generated_samples.py --results-root /home/zheyuanlai/levy-sampling-production/results --output-root /home/zheyuanlai/levy-sampling-production/figures --cache-root /home/zheyuanlai/levy-sampling-production/cache/generated_samples --manifest-path /home/zheyuanlai/levy-sampling-production/cache/generated_samples/generated_sample_plots_manifest.json --overwrite


double_well: wrote 2 figure pairs from results/double_well/positions.csv
mog40: wrote 2 figure pairs from results/mog40/positions.csv
mb3well_10d: wrote 2 figure pairs from results/mb3well_10d/positions.csv
coupled_phi4: wrote 2 figure pairs from results/coupled_phi4/positions.csv
manifest: /home/zheyuanlai/levy-sampling-production/cache/generated_samples/generated_sample_plots_manifest.json


CompletedProcess(args=['/home/zheyuanlai/miniconda3/envs/jcp-levy-release/bin/python', '/home/zheyuanlai/levy-sampling-production/scripts/replot_generated_samples.py', '--results-root', '/home/zheyuanlai/levy-sampling-production/results', '--output-root', '/home/zheyuanlai/levy-sampling-production/figures', '--cache-root', '/home/zheyuanlai/levy-sampling-production/cache/generated_samples', '--manifest-path', '/home/zheyuanlai/levy-sampling-production/cache/generated_samples/generated_sample_plots_manifest.json', '--overwrite'], returncode=0)

In [5]:
report = validate_release(
    ROOT,
    check_results=True,
    require_figures=True,
)
from src.manuscript import FIGURE_FORMATS

print("Final validation:", report["status"])
for _ext in FIGURE_FORMATS:
    _files = sorted((ROOT / "figures" / _ext).glob(f"*.{_ext}"))
    print(f"figures/{_ext}: {len(_files)} files")

Final validation: passed
figures/png: 68 files
figures/tiff: 60 files
figures/svg: 60 files
figures/pdf: 68 files
